# TerraAlert — Flood Dataset EDA
**Dataset:** NOAA Storm Events Database (2018–2024)  
**Research Question:** Does transfer learning with ResNet-50 outperform a baseline CNN for flood extent detection from satellite imagery?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Download NOAA Storm Events Data

In [ ]:
# NOAA Storm Events — download multiple years and filter for floods
# Direct CSV download, no API key needed

years = [2019, 2020, 2021, 2022, 2023]
all_dfs = []

for year in years:
    url = f'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d{year}_c20240716.csv.gz'
    print(f'Downloading {year}...')
    try:
        df_year = pd.read_csv(url, compression='gzip', low_memory=False)
        # Filter for flood events only
        flood_df = df_year[df_year['EVENT_TYPE'].isin([
            'Flash Flood', 'Flood', 'Coastal Flood', 'Lakeshore Flood'
        ])]
        all_dfs.append(flood_df)
        print(f'  {year}: {len(flood_df):,} flood events')
    except Exception as e:
        print(f'  {year}: Error — {e}')
        print(f'  Manual download: https://www.ncdc.noaa.gov/stormevents/ftp.jsp')

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    df.to_csv('../data/floods/noaa_floods.csv', index=False)
    print(f'\nTotal flood records: {len(df):,} ✅')
else:
    print('\nManual download required — see instructions above')

In [ ]:
df = pd.read_csv('../data/floods/noaa_floods.csv', low_memory=False)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns[:15])}...')
df.head()

## 2. Data Quality Check

In [ ]:
print('=== DATASET OVERVIEW ===')
print(f'Total records: {len(df):,}')
print(f'\nFlood types:')
print(df['EVENT_TYPE'].value_counts())
print(f'\nNull values (key columns):')
key_cols = ['EVENT_TYPE', 'STATE', 'DEATHS_DIRECT', 'DAMAGE_PROPERTY', 'BEGIN_LAT', 'BEGIN_LON']
existing = [c for c in key_cols if c in df.columns]
print(df[existing].isnull().sum())

## 3. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('TerraAlert — Flood Dataset EDA', fontsize=16, fontweight='bold')

# Plot 1: Flood type distribution
flood_types = df['EVENT_TYPE'].value_counts()
axes[0,0].bar(flood_types.index, flood_types.values, color='#3498db', edgecolor='white')
axes[0,0].set_title('Flood Event Types')
axes[0,0].set_xlabel('Type')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=15)

# Plot 2: Events per year
if 'YEAR' in df.columns:
    yearly = df.groupby('YEAR').size()
    axes[0,1].bar(yearly.index, yearly.values, color='#2980b9', edgecolor='white')
    axes[0,1].set_title('Flood Events Per Year')
    axes[0,1].set_xlabel('Year')
    axes[0,1].set_ylabel('Count')

# Plot 3: Top 10 states
if 'STATE' in df.columns:
    top_states = df['STATE'].value_counts().head(10)
    axes[1,0].barh(top_states.index, top_states.values, color='#1abc9c')
    axes[1,0].set_title('Top 10 States by Flood Events')
    axes[1,0].set_xlabel('Count')

# Plot 4: Deaths distribution
if 'DEATHS_DIRECT' in df.columns:
    deaths = df['DEATHS_DIRECT'].fillna(0)
    deaths_nonzero = deaths[deaths > 0]
    axes[1,1].hist(deaths_nonzero, bins=20, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[1,1].set_title('Direct Deaths Per Event (non-zero)')
    axes[1,1].set_xlabel('Deaths')
    axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/floods/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved ✅')

## 4. Model Justification

**For this flood dataset, I will compare:**

| Model | Data Type | Reason |
|---|---|---|
| **Random Forest** | Tabular (NOAA) | Baseline for structured flood event prediction |
| **XGBoost** | Tabular (NOAA) | Gradient boosting upgrade, better handles damage cost prediction |
| **Basic CNN** | Satellite imagery (Copernicus) | Baseline image classifier for flood extent detection |
| **ResNet-50 (Transfer Learning)** | Satellite imagery (Copernicus) | Pretrained on ImageNet, fine-tuned for flood segmentation |

**Target variable (tabular):** Flood severity class (based on deaths + damage)  
**Target variable (image):** Binary flood/no-flood pixel classification  
**Evaluation metrics:** F1-score, AUC-ROC, IoU (for image segmentation)  

**Copernicus satellite data:** Download at https://emergency.copernicus.eu/mapping  
→ Search for activation type 'Flood', download GeoTIFF files


In [ ]:
# Feature engineering for tabular flood prediction
def parse_damage(val):
    if pd.isna(val): return 0
    val = str(val).upper().strip()
    if val.endswith('K'): return float(val[:-1]) * 1e3
    if val.endswith('M'): return float(val[:-1]) * 1e6
    if val.endswith('B'): return float(val[:-1]) * 1e9
    try: return float(val)
    except: return 0

if 'DAMAGE_PROPERTY' in df.columns:
    df['damage_numeric'] = df['DAMAGE_PROPERTY'].apply(parse_damage)
    df['deaths_total'] = df['DEATHS_DIRECT'].fillna(0) + df.get('DEATHS_INDIRECT', pd.Series(0, index=df.index)).fillna(0)
    
    # Severity score: 0=low, 1=medium, 2=high
    df['severity'] = 0
    df.loc[df['damage_numeric'] > 10000, 'severity'] = 1
    df.loc[(df['damage_numeric'] > 1000000) | (df['deaths_total'] > 0), 'severity'] = 2
    
    print('Severity distribution:')
    print(df['severity'].value_counts())
    
    feature_cols = ['BEGIN_LAT', 'BEGIN_LON', 'damage_numeric', 'deaths_total']
    available = [c for c in feature_cols if c in df.columns]
    df_model = df[available + ['severity']].dropna()
    df_model.to_csv('../data/floods/floods_processed.csv', index=False)
    print(f'\nModel-ready shape: {df_model.shape} ✅')